In [1]:
import pandas as pd

In [26]:
# 1. Load raw data
df = pd.read_csv(
    "../01_raw_data/DataCoSupplyChainDataset.csv",
    encoding="latin1"
)

In [14]:
print("Original shape:", df.shape)

Original shape: (180519, 53)


In [27]:
# 2. Remove unwanted and duplicate columns
columns_to_drop = [
    "Customer Email",
    "Customer Password",
    "Product Description",
    "Product Image",
    "Product Status",
    "Order Zipcode",
    "Order Customer Id",
    "Product Category Id",
    "Order Item Cardprod Id",
    "Sales per customer",
    "Benefit per order",
    "Order Item Product Price"
]
df = df.drop(columns=columns_to_drop)
print("Shape after removing columns:", df.shape)

Shape after removing columns: (180519, 41)


""" 
df.columns = (
    df.columns
      .str.strip()                       # remove leading/trailing spaces
      .str.lower()                       # convert to lowercase
      .str.replace(" ", "_")             # replace spaces with underscores
      .str.replace(r"[^\w]", "", regex=True)  # remove special characters
)
df.columns
"""

In [28]:
# 3. Rename columns
df = df.rename(columns={
    "Type": "payment_type",
    "Days for shipping (real)": "days_for_shipping_real",
    "Days for shipment (scheduled)": "days_for_shipment_scheduled",
    "Delivery Status": "delivery_status",
    "Late_delivery_risk": "late_delivery_risk",
    
    "Category Id": "category_id",
    "Category Name": "category_name",
    
    "Customer City": "customer_city",
    "Customer Country": "customer_country",
    "Customer Fname": "customer_first_name",
    "Customer Id": "customer_id",
    "Customer Lname": "customer_last_name",
    "Customer Segment": "customer_segment",
    "Customer State": "customer_state",
    "Customer Street": "customer_street",
    "Customer Zipcode": "customer_zipcode",
   
    "Department Id": "department_id",
    "Department Name": "department_name",

    "Latitude": "latitude",
    "Longitude": "longitude",
    "Market": "market",

    "Order City": "order_city",
    "Order Country": "order_country",
    "order date (DateOrders)": "order_date",
    "Order Id": "order_id",

    "Order Item Discount": "order_item_discount",
    "Order Item Discount Rate": "order_item_discount_rate",
    "Order Item Id": "order_item_id",
    "Order Item Profit Ratio": "order_item_profit_ratio",
    "Order Item Quantity": "order_item_quantity",

    "Sales": "sales",
    "Order Item Total": "order_item_total",
    "Order Profit Per Order": "order_profit_per_order",

    "Order Region": "order_region",
    "Order State": "order_state",
    "Order Status": "order_status",

    "Product Card Id": "product_card_id",
    "Product Name": "product_name",
    "Product Price": "product_price",

    "shipping date (DateOrders)": "shipping_date",
    "Shipping Mode": "shipping_mode"
})


# 4. Check the new column names
print("\nCleaned column names:")
print(df.columns.tolist())


Cleaned column names:
['payment_type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_first_name', 'customer_id', 'customer_last_name', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_date', 'order_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_name', 'product_price', 'shipping_date', 'shipping_mode']


In [29]:
# 5. Convert date columns to datetime
df["order_date"] = pd.to_datetime(df["order_date"])
df["shipping_date"] = pd.to_datetime(df["shipping_date"])

print("\nDate data types:")
print(df[["order_date", "shipping_date"]].dtypes)


Date data types:
order_date       datetime64[us]
shipping_date    datetime64[us]
dtype: object


In [30]:
# 6. Calculate shipping variance
df["shipping_variance_days"] = (
    df["days_for_shipping_real"]
    - df["days_for_shipment_scheduled"]
)

print("\nShipping variance:")
print(df["shipping_variance_days"].describe())


Shipping variance:
count    180519.000000
mean          0.565807
std           1.490966
min          -2.000000
25%           0.000000
50%           1.000000
75%           1.000000
max           4.000000
Name: shipping_variance_days, dtype: float64


In [31]:
# 7. Create SLA breach flag
df["sla_breach_flag"] = (
    df["shipping_variance_days"] > 0
).astype(int)

print("\nSLA breach flag:")
print(df["sla_breach_flag"].value_counts())


SLA breach flag:
sla_breach_flag
1    103400
0     77119
Name: count, dtype: int64


In [35]:
# 8. Final validation

print("\nFinal shape:")
print(df.shape)

print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nFinal columns:")
print(df.columns.tolist())


Final shape:
(180519, 43)

Missing values:
customer_last_name    8
customer_zipcode      3
dtype: int64

Duplicate rows:
0

Final columns:
['payment_type', 'days_for_shipping_real', 'days_for_shipment_scheduled', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_first_name', 'customer_id', 'customer_last_name', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_date', 'order_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'product_card_id', 'product_name', 'product_price', 'shipping_date', 'shipping_mode', 'shipping_variance_days', 'sla_breach_flag']


In [38]:
# 9. Export cleaned dataset

output_path = "../Output/dataco_cleaned.csv"

df.to_csv(output_path, index=False)

print("\nCleaned dataset exported successfully:")
print(output_path)


Cleaned dataset exported successfully:
../Output/dataco_cleaned.csv
